In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from algorithms.sort import Sort
frame_buffer = []

cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
model = YOLO('/home/max/Desktop/tennis/tennis-v5/models/yolo/yolo11n.pt')
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)


while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_buffer.append(frame)
    if len(frame_buffer) == 3:
        for frame in frame_buffer:
            results_list = model(frame, classes=0, conf=0.3, verbose=False)
            for result in results_list:
                boxes_xyxy = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()
                for box, conf in zip(boxes_xyxy, confs):
                    x1, y1, x2, y2 = box.astype(int)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                if len(boxes_xyxy) > 0:
                    detections = np.concatenate((boxes_xyxy, confs), axis=1)
                    tracked = tracker.update(detections, np.array([]))
                else:
                    tracked = tracker.update(np.empty((0, 5)), np.array([]))
            cv2.imshow('frame', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
        frame_buffer = []
cap.release()
cv2.destroyAllWindows()

In [10]:
import cv2
import numpy as np
from ultralytics import YOLO
from algorithms.sort import Sort

cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
model = YOLO('/home/max/Desktop/tennis/tennis-v5/models/yolo/yolo11n.pt')
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)
frame_number = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_number += 1
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC)  # 获取毫秒时间戳
    
    # YOLO检测
    results = model(frame, classes=0, conf=0.3, verbose=False)[0]
    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy().reshape(-1, 1)
    
    # SORT追踪
    if boxes_xyxy.shape[0] > 0:
        detections = np.hstack((boxes_xyxy, confs))
    else:
        detections = np.empty((0, 5))
    
    tracked = tracker.update(detections, np.array([]))
    
    # 终端输出
    if tracked.size == 0:
        # 无检测结果时输出全零
        print(f"{timestamp:.0f},{frame_number},0,0,0,0,0,0,0,0")
    else:
        for obj in tracked:
            x1, y1, x2, y2 = map(int, obj[:4])
            points.extend([x1, y1, x2, y2])
            # 生成矩形四顶点坐标（顺时针顺序）
            points = [
                x1, y1,  # 左上
                x2, y1,  # 右上
                x2, y2,  # 右下
                x1, y2   # 左下
            ]
            print(f"{timestamp:.0f},{frame_number},{','.join(map(str, points))}")

cap.release()

0,1,573,223,594,223,594,269,573,269
0,1,993,478,1059,478,1059,662,993,662
40,2,573,223,594,223,594,270,573,270
40,2,993,477,1057,477,1057,659,993,659
80,3,572,222,594,222,594,270,572,270
80,3,993,474,1057,474,1057,657,993,657
120,4,572,223,594,223,594,270,572,270
120,4,994,473,1057,473,1057,654,994,654
160,5,572,223,593,223,593,270,572,270
160,5,995,472,1056,472,1056,651,995,651
200,6,572,223,594,223,594,270,572,270
200,6,994,472,1056,472,1056,649,994,649
240,7,572,223,594,223,594,270,572,270
240,7,995,470,1056,470,1056,645,995,645
280,8,572,223,594,223,594,270,572,270
280,8,995,470,1056,470,1056,641,995,641
320,9,572,223,594,223,594,270,572,270
320,9,995,470,1056,470,1056,639,995,639
360,10,571,223,594,223,594,270,571,270
360,10,996,471,1056,471,1056,639,996,639
400,11,571,223,594,223,594,270,571,270
400,11,996,473,1055,473,1055,639,996,639
440,12,571,223,595,223,595,270,571,270
440,12,996,474,1056,474,1056,639,996,639
480,13,571,223,594,223,594,270,571,270
480,13,995,475,1056,475,105

In [11]:
import cv2
import numpy as np
from ultralytics import YOLO
from algorithms.sort import Sort

cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
model = YOLO('/home/max/Desktop/tennis/tennis-v5/models/yolo/yolo11n.pt')
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)
frame_number = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_number += 1
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC)
    
    # YOLO检测
    results = model(frame, classes=0, conf=0.3, verbose=False)[0]
    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy().reshape(-1, 1)
    
    # SORT追踪
    if boxes_xyxy.shape[0] > 0:
        detections = np.hstack((boxes_xyxy, confs))
    else:
        detections = np.empty((0, 5))
    
    tracked = tracker.update(detections, np.array([]))
    
    # 构建输出行
    output = [f"{timestamp:.0f}", str(frame_number)]
    
    if tracked.size == 0:
        output.extend(['0']*4)  # 无检测时填充四个0
    else:
        for obj in tracked:
            x1, y1, x2, y2 = map(str, map(int, obj[:4]))
            output.extend([x1, y1, x2, y2])
    
    # 输出同一帧的所有结果
    print(','.join(output))

0,1,573,223,594,269,993,478,1059,662
40,2,573,223,594,270,993,477,1057,659
80,3,572,222,594,270,993,474,1057,657
120,4,572,223,594,270,994,473,1057,654
160,5,572,223,593,270,995,472,1056,651
200,6,572,223,594,270,994,472,1056,649
240,7,572,223,594,270,995,470,1056,645
280,8,572,223,594,270,995,470,1056,641
320,9,572,223,594,270,995,470,1056,639
360,10,571,223,594,270,996,471,1056,639
400,11,571,223,594,270,996,473,1055,639
440,12,571,223,595,270,996,474,1056,639
480,13,571,223,594,270,995,475,1056,640
520,14,571,223,594,270,993,476,1056,641
560,15,571,223,594,269,989,476,1056,643
600,16,571,223,595,269,986,477,1056,644
640,17,984,477,1057,645
680,18,982,476,1057,645
720,19,980,476,1057,645
760,20,977,477,1056,647
800,21,973,477,1054,648
840,22,969,479,1053,651
880,23,965,481,1052,655
920,24,962,485,1053,661
960,25,960,488,1053,667
1000,26,959,490,1053,670
1040,27,959,491,1052,672
1080,28,957,490,1049,672
1120,29,956,488,1047,670
1160,30,956,486,1045,669
1200,31,585,222,608,268,956,485,

In [12]:
import cv2
import numpy as np
from ultralytics import YOLO
from algorithms.sort import Sort
from collections import OrderedDict
from filterpy.kalman import KalmanFilter

class PlayerTracker:
    def __init__(self):
        self.players = OrderedDict()  # 保持插入顺序
        self.kalman_filters = {}
        self.max_id = 0
        self.lost_threshold = 5  # 最大丢失帧数

    def init_kalman(self):
        """初始化卡尔曼滤波器"""
        kf = KalmanFilter(dim_x=4, dim_z=2)
        dt = 1.0  # 时间间隔
        
        # 状态转移矩阵 (假设匀速运动)
        kf.F = np.array([[1, 0, dt, 0],
                         [0, 1, 0, dt],
                         [0, 0, 1, 0],
                         [0, 0, 0, 1]])
        
        # 观测矩阵
        kf.H = np.array([[1, 0, 0, 0],
                         [0, 1, 0, 0]])
        
        # 协方差矩阵
        kf.P *= 100
        kf.R = np.diag([5, 5])  # 观测噪声
        kf.Q = np.eye(4) * 0.1   # 过程噪声
        return kf

    def update_players(self, detections):
        """更新球员跟踪信息"""
        current_ids = []
        
        # 按y2坐标降序排列 (y值越大位置越靠下)
        sorted_det = sorted(detections, key=lambda x: x[3], reverse=True)
        
        for box in sorted_det:
            x1, y1, x2, y2 = map(int, box[:4])
            cx, cy = (x1+x2)//2, (y1+y2)//2  # 中心坐标
            
            # 寻找最近已有ID
            min_dist = float('inf')
            assigned_id = None
            for pid, (last_pos, lost_count) in self.players.items():
                dist = np.linalg.norm(np.array(last_pos) - np.array([cx, cy]))
                if dist < min_dist and dist < 50:  # 距离阈值50像素
                    min_dist = dist
                    assigned_id = pid
            
            # 分配新ID或更新现有ID
            if assigned_id is None:
                self.max_id += 1
                assigned_id = self.max_id
                self.kalman_filters[assigned_id] = self.init_kalman()
                self.kalman_filters[assigned_id].x = np.array([cx, cy, 0, 0])
            else:
                # 更新卡尔曼滤波器
                kf = self.kalman_filters[assigned_id]
                kf.predict()
                kf.update(np.array([cx, cy]))
            
            # 更新球员信息
            self.players[assigned_id] = ((cx, cy), 0)
            current_ids.append(assigned_id)
        
        # 处理丢失的ID
        lost_ids = set(self.players.keys()) - set(current_ids)
        for pid in lost_ids:
            pos, lost_count = self.players[pid]
            if lost_count >= self.lost_threshold:
                del self.players[pid]
                del self.kalman_filters[pid]
            else:
                # 使用卡尔曼预测
                kf = self.kalman_filters[pid]
                kf.predict()
                pred_pos = kf.x[:2].astype(int)
                self.players[pid] = (pred_pos, lost_count+1)

    def get_output(self, frame_size):
        """生成格式化的输出行"""
        output = []
        for pid in sorted(self.players.keys()):  # 按ID排序
            (cx, cy), lost_count = self.players[pid]
            
            # 当丢失超过阈值时输出全零
            if lost_count > 0:
                output.extend([0]*4)
            else:
                # 生成边界框（示例使用固定大小）
                box_size = 100  # 可根据需求调整
                x1 = max(0, cx - box_size//2)
                y1 = max(0, cy - box_size//2)
                x2 = min(frame_size[1], cx + box_size//2)
                y2 = min(frame_size[0], cy + box_size//2)
                output.extend([x1, y1, x2, y2])
        
        # 补全4人位置（如果不足）
        while len(output) < 16:  # 4人*4坐标
            output.extend([0]*4)
        
        return output[:16]  # 最多4人

# 初始化组件
cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
model = YOLO('/home/max/Desktop/tennis/tennis-v5/models/yolo/yolo11n.pt')
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)
player_tracker = PlayerTracker()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # 获取视频信息
    frame_number = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC)
    h, w = frame.shape[:2]
    
    # YOLO检测
    results = model(frame, classes=0, conf=0.3, verbose=False)[0]
    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    
    # SORT追踪
    if boxes_xyxy.shape[0] > 0:
        detections = np.hstack((boxes_xyxy, np.ones((len(boxes_xyxy),1))))
    else:
        detections = np.empty((0, 5))
    
    tracked = tracker.update(detections, np.array([]))
    
    # 更新球员跟踪
    player_tracker.update_players(tracked)
    
    # 生成输出
    output = player_tracker.get_output((h, w))
    output_str = f"{timestamp:.0f},{frame_number}," + ",".join(map(str, output))
    print(output_str)

cap.release()

0,1,976,520,1076,620,533,196,633,296,0,0,0,0,0,0,0,0
40,2,975,518,1075,618,533,196,633,296,0,0,0,0,0,0,0,0
80,3,975,515,1075,615,533,196,633,296,0,0,0,0,0,0,0,0
120,4,975,513,1075,613,533,196,633,296,0,0,0,0,0,0,0,0
160,5,975,511,1075,611,532,196,632,296,0,0,0,0,0,0,0,0
200,6,975,510,1075,610,533,196,633,296,0,0,0,0,0,0,0,0
240,7,975,507,1075,607,533,196,633,296,0,0,0,0,0,0,0,0
280,8,975,505,1075,605,533,196,633,296,0,0,0,0,0,0,0,0
320,9,975,504,1075,604,533,196,633,296,0,0,0,0,0,0,0,0
360,10,976,505,1076,605,532,196,632,296,0,0,0,0,0,0,0,0
400,11,975,506,1075,606,532,196,632,296,0,0,0,0,0,0,0,0
440,12,976,506,1076,606,533,196,633,296,0,0,0,0,0,0,0,0
480,13,975,507,1075,607,532,196,632,296,0,0,0,0,0,0,0,0
520,14,974,508,1074,608,532,196,632,296,0,0,0,0,0,0,0,0
560,15,972,509,1072,609,532,196,632,296,0,0,0,0,0,0,0,0
600,16,971,510,1071,610,533,196,633,296,0,0,0,0,0,0,0,0
640,17,970,511,1070,611,0,0,0,0,0,0,0,0,0,0,0,0
680,18,969,510,1069,610,0,0,0,0,0,0,0,0,0,0,0,0
720,19,968,510,1068,6

In [15]:
import cv2
import numpy as np
from ultralytics import YOLO
from algorithms.sort import Sort
from collections import deque, OrderedDict
from filterpy.kalman import KalmanFilter

class EnhancedPlayerTracker:
    def __init__(self):
        self.players = OrderedDict()  # 固定ID顺序
        self.trajectories = {}        # 轨迹历史记录
        self.kalman_filters = {}
        self.max_id = 0
        self.lost_threshold = 15      # 丢失帧数阈值
        self.traj_length = 5         # 轨迹历史长度

    def init_kalman(self):
        """初始化自适应卡尔曼滤波器"""
        kf = KalmanFilter(dim_x=6, dim_z=2)  # 状态：x,y,vx,vy,ax,ay
        dt = 1.0  # 时间间隔
        
        # 自适应状态转移矩阵
        kf.F = np.array([
            [1, 0, dt, 0, 0.5*dt**2, 0],
            [0, 1, 0, dt, 0, 0.5*dt**2],
            [0, 0, 1, 0, dt, 0],
            [0, 0, 0, 1, 0, dt],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 1]
        ])
        
        # 观测矩阵
        kf.H = np.array([[1, 0, 0, 0, 0, 0],
                         [0, 1, 0, 0, 0, 0]])
        
        # 初始化协方差矩阵
        kf.P = np.eye(6) * 100
        kf.R = np.diag([10, 10])     # 观测噪声
        kf.Q = np.eye(6) * 0.1       # 过程噪声
        return kf

    def calculate_dynamics(self, pid):
        """基于轨迹历史计算实际运动参数"""
        trajectory = self.trajectories.get(pid, [])
        if len(trajectory) < 2:
            return 0, 0, 0, 0  # vx, vy, ax, ay
        
        # 计算速度
        dx = np.diff([p[0] for p in trajectory])
        dy = np.diff([p[1] for p in trajectory])
        vx = np.mean(dx[-3:]) if len(dx)>=3 else dx[-1] if dx else 0
        vy = np.mean(dy[-3:]) if len(dy)>=3 else dy[-1] if dy else 0
        
        # 计算加速度
        if len(trajectory) >=3:
            ddx = np.diff(dx)
            ddy = np.diff(dy)
            ax = np.mean(ddx[-2:]) if ddx else 0
            ay = np.mean(ddy[-2:]) if ddy else 0
        else:
            ax, ay = 0, 0
            
        return vx, vy, ax, ay

    def update_kalman(self, pid, measurement=None):
        """更新卡尔曼滤波器"""
        kf = self.kalman_filters[pid]
        
        if measurement is None:  # 预测模式
            # 从轨迹历史获取动态参数
            vx, vy, ax, ay = self.calculate_dynamics(pid)
            
            # 更新状态转移矩阵
            dt = 1.0
            kf.F = np.array([
                [1, 0, dt, 0, 0.5*dt**2, 0],
                [0, 1, 0, dt, 0, 0.5*dt**2],
                [0, 0, 1, 0, dt, 0],
                [0, 0, 0, 1, 0, dt],
                [0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 1]
            ])
            
            # 注入计算的加速度
            kf.x[4] = ax
            kf.x[5] = ay
            
            kf.predict()
        else:  # 更新模式
            kf.update(measurement)
            
        return kf.x[:2]  # 返回预测/更新的位置

    def update_players(self, detections, frame_size):
        """更新球员状态"""
        # 按初始y2位置分配ID（仅首次）
        if not self.players and len(detections) > 0:
            sorted_det = sorted(detections, key=lambda x: x[3], reverse=True)
            for box in sorted_det:
                self._assign_new_id(box, frame_size)
            return
                
        # 跟踪匹配逻辑
        current_ids = []
        for box in detections:
            # 寻找最近已有轨迹
            min_dist = float('inf')
            matched_id = None
            current_pos = ((box[0]+box[2])/2, (box[1]+box[3])/2)
            
            for pid in self.players:
                # 获取预测位置
                pred_pos = self.update_kalman(pid)  # 先预测
                dist = np.linalg.norm(pred_pos - current_pos)
                
                if dist < min(dist, 100):  # 动态阈值
                    min_dist = dist
                    matched_id = pid
                    
            if matched_id:
                # 更新已有ID
                self._update_id(matched_id, box, current_pos)
                current_ids.append(matched_id)
            else:
                # 分配新ID（仅当有位置空缺时）
                if len(self.players) < 4:
                    self._assign_new_id(box, frame_size)
                    current_ids.append(self.max_id)

        # 处理丢失的ID
        for pid in list(self.players.keys()):
            if pid not in current_ids:
                self._handle_lost_id(pid, frame_size)

    def _assign_new_id(self, box, frame_size):
        """分配新ID并初始化跟踪器"""
        self.max_id += 1
        cx, cy = (box[0]+box[2])//2, (box[1]+box[3])//2
        self.players[self.max_id] = {
            'position': (cx, cy),
            'lost_count': 0,
            'history': deque(maxlen=self.traj_length)
        }
        self.trajectories[self.max_id] = deque(maxlen=self.traj_length)
        self.kalman_filters[self.max_id] = self.init_kalman()
        self.kalman_filters[self.max_id].x[:2] = [cx, cy]

    def get_output(self):
        """生成固定格式的输出"""
        output = []
        for pid in self.players:  # 按固定ID顺序输出
            player = self.players[pid]
            
            if player['lost_count'] > self.lost_threshold:
                output.extend([0]*4)
            else:
                # 使用卡尔曼预测位置生成边界框
                pred_pos = self.update_kalman(pid)  # 预测
                box_size = self._get_adaptive_size(pid)
                x1 = int(max(0, pred_pos[0] - box_size[0]//2))
                y1 = int(max(0, pred_pos[1] - box_size[1]//2))
                x2 = int(min(frame_size[1], pred_pos[0] + box_size[0]//2))
                y2 = int(min(frame_size[0], pred_pos[1] + box_size[1]//2))
                output.extend([x1, y1, x2, y2])
        
        # 补全到4人
        while len(output) < 16:
            output.extend([0]*4)
            
        return output

# 初始化组件
cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
model = YOLO('/home/max/Desktop/tennis/tennis-v5/models/yolo/yolo11n.pt')
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)
player_tracker = EnhancedPlayerTracker()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # 获取视频信息
    frame_number = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC)
    h, w = frame.shape[:2]
    
    # YOLO检测
    results = model(frame, classes=0, conf=0.3, verbose=False)[0]
    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    
    # SORT追踪
    if boxes_xyxy.shape[0] > 0:
        detections = np.hstack((boxes_xyxy, np.ones((len(boxes_xyxy),1))))
    else:
        detections = np.empty((0, 5))
    
    tracked = tracker.update(detections, np.array([]))
    
    # 更新跟踪器
    player_tracker.update_players(tracked, (h, w))
    
    # 生成输出
    output = player_tracker.get_output()
    print(f"{timestamp:.0f},{frame_number}," + ",".join(map(str, output)))

cap.release()

ValueError: could not broadcast input array from shape (2,) into shape (2,1)

In [17]:
import cv2
import numpy as np
import os
import sys
from algorithms.sort import Sort
from ultralytics import YOLO

class PeopleTracker:
    def __init__(self, model_path="/home/max/Desktop/tennis/tennis-v5/weights/yolo11x.pt", target_people=2, roi_expansion=0.3):
        # 初始化YOLO模型
        self.model = YOLO(model_path)
        self.model_params = {
            'classes': [0],     # 只检测人物
            'conf': 0.3,        # 置信度阈值
            'verbose': False    
        }
        
        # 初始化SORT跟踪器
        self.sort_tracker = Sort(max_age=15, min_hits=3, iou_threshold=0.3)
        
        # 配置运行参数
        self.target_people = target_people
        self.roi_expansion = roi_expansion
        self.current_roi = None    # 当前有效ROI区域（合并后的单一区域）
        self.track_history = {}    # 跟踪状态记录

    def process_frame(self, frame):
        # 决策检测模式
        if not self.current_roi:
            detections = self._full_detection(frame)
        else:
            detections = self._roi_detection(frame)
            # 当ROI检测失败时回退全图检测
            if detections.size == 0:
                self.current_roi = None
                detections = self._full_detection(frame)

        # 更新跟踪器
        tracked_objs = self.sort_tracker.update(np.array(detections) if detections.size else np.empty((0,5)), np.array([]))
        
        # 更新系统状态
        self._update_tracking_context(tracked_objs, frame.shape[1], frame.shape[0])
        
        return self._format_output(tracked_objs)

    def _full_detection(self, frame):
        """全图检测模式"""
        results = self.model.predict(frame, **self.model_params)
        return self._parse_detections(results)

    def _roi_detection(self, frame):
        """ROI区域检测"""
        x1, y1, x2, y2 = self.current_roi
        roi_img = frame[y1:y2, x1:x2]
        
        if roi_img.size == 0:
            return np.empty((0, 5))
            
        results = self.model.predict(roi_img, **self.model_params)
        detections = []
        for det in self._parse_detections(results):
            # 坐标转换到原图
            det[0] += x1
            det[1] += y1
            det[2] += x1
            det[3] += y1
            detections.append(det)
        
        return np.array(detections) if detections else np.empty((0, 5))

    def _parse_detections(self, results):
        """解析检测结果"""
        detections = []
        for result in results:
            for box in result.boxes:
                xyxy = box.xyxy.cpu().numpy()[0]
                conf = box.conf.item()
                detections.append([*xyxy, conf])
        return np.array(detections) if detections else np.empty((0, 5))

    def _update_tracking_context(self, tracked_objs, img_w, img_h):
        """更新跟踪上下文"""
        # 维护跟踪历史
        current_ids = set(int(obj[4]) for obj in tracked_objs)
        
        # 清理丢失的目标
        self.track_history = {
            k: v for k, v in self.track_history.items() 
            if k in current_ids
        }
        
        # 更新现有目标状态
        individual_rois = []
        for obj in tracked_objs:
            obj_id = int(obj[4])
            # 记录个体ROI
            individual_roi = self._expand_roi(obj[:4], img_w, img_h)
            individual_rois.append(individual_roi)
            
            if obj_id not in self.track_history:
                self.track_history[obj_id] = {
                    'age': 0,
                    'trajectory': []
                }
            self.track_history[obj_id]['age'] += 1
            self.track_history[obj_id]['trajectory'].append(obj[:4])
        
        # 生成合并后的全局ROI
        if individual_rois:
            self.current_roi = self._merge_rois(individual_rois, img_w, img_h)
        else:
            self.current_roi = None

    def _expand_roi(self, bbox, img_w, img_h):
        """生成单个目标的扩展ROI"""
        x1, y1, x2, y2 = map(int, bbox)
        w = x2 - x1
        h = y2 - y1
        
        return [
            max(0, x1 - int(w * self.roi_expansion)),
            max(0, y1 - int(h * self.roi_expansion)),
            min(img_w, x2 + int(w * self.roi_expansion)),
            min(img_h, y2 + int(h * self.roi_expansion))
        ]

    def _merge_rois(self, rois, img_w, img_h):
        """合并多个ROI为单一区域"""
        # 计算合并后的边界
        min_x = min(r[0] for r in rois)
        min_y = min(r[1] for r in rois)
        max_x = max(r[2] for r in rois)
        max_y = max(r[3] for r in rois)
        
        # 应用安全边界
        return [
            max(0, min_x),
            max(0, min_y),
            min(img_w, max_x),
            min(img_h, max_y)
        ]

    def _format_output(self, tracked_objs):
        """生成最终输出"""
        # 按跟踪稳定性排序
        sorted_objs = sorted(
            tracked_objs,
            key=lambda x: self.track_history.get(int(x[4]), {'age': 0})['age'],
            reverse=True
        )
        
        # 提取目标框
        selected = [list(map(int, obj[:4])) for obj in sorted_objs[:self.target_people]]
        
        # 使用历史数据补足数量
        if len(selected) < self.target_people:
            missing = self.target_people - len(selected)
            candidates = sorted(
                self.track_history.values(),
                key=lambda x: x['age'],
                reverse=True
            )[:missing]
            selected.extend(
                list(map(int, c['trajectory'][-1]))
                for c in candidates if c['trajectory']
            )
        
        return selected[:self.target_people]

# 使用示例
if __name__ == "__main__":
    tracker = PeopleTracker(
        model_path="/home/max/Desktop/tennis/tennis-v5/weights/yolo11x.pt",
        target_people=2,
        roi_expansion=0.4
    )

    cap = cv2.VideoCapture("/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        bboxes = tracker.process_frame(frame)
        
        # 可视化合并后的ROI
        if tracker.current_roi:
            x1, y1, x2, y2 = tracker.current_roi
            cv2.rectangle(frame, (x1,y1), (x2,y2), (255,0,0), 2)
        
        # 绘制追踪框
        for bbox in bboxes:
            x1, y1, x2, y2 = bbox
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        
        cv2.imshow('Tracking', frame)
        if cv2.waitKey(1) == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [21]:
import cv2
import numpy as np
import os
import sys

sys.path.append('/home/max/Desktop/tennis/tennis-v5')

from ultralytics import YOLO
from algorithms.sort import Sort
from utils.shared_data import SharedData
import torchvision.transforms as transforms

import time

class PlayerTracker(SharedData):
    def __init__(self):
        super().__init__()
        # 初始化模型组件
        self.detector = YOLO(self.yolo_path).to(self.device)
        self.tracker = Sort(max_age=10, min_hits=3, iou_threshold=0.3)
        
        # 配置参数
        self.roi_expansion = 0.3
        self.history_size = 10
        self.frame_buffer = 25
        
        # 状态存储
        self.current_roi = None
        
        self.track_history = {}  # {track_id: {'positions': [], 'velocities': [], 'accelerations': []}}
        
        self.timing_stats = {
            "initialize_roi": [],
            "roi_crop": [],
            "model_inference": [],
            "convert_coordinates": [],
            "sort_tracking": [],
            "update_history": [],
            "calculate_target_roi": [],
            "merge_rois": [],
        }

    def _initialize_roi(self):
        """初始化ROI为全帧"""
        self.current_roi = np.array([[0, 0, self.frame_width, self.frame_height]])

    def _update_history(self, track_id: int, bbox: np.ndarray):
        """
        更新指定track_id的运动历史数据
        :param track_id: 目标ID
        :param bbox: 当前边界框 [x1,y1,x2,y2]
        """
        if track_id not in self.track_history:
            self.track_history[track_id] = {
                'positions': [],
                'velocities': [],
                'accelerations': []
            }
        
        history = self.track_history[track_id]
        
        # 更新位置数据
        history['positions'].append(bbox.copy())
        if len(history['positions']) > self.history_size:
            history['positions'].pop(0)
        
        # 计算速度
        if len(history['positions']) >= 2:
            velocity = bbox - history['positions'][-2]
            history['velocities'].append(velocity)
            if len(history['velocities']) > self.history_size:
                history['velocities'].pop(0)
        
        # 计算加速度
        if len(history['positions']) >= 3:
            acc = history['positions'][-1] - 2*history['positions'][-2] + history['positions'][-3]
            history['accelerations'].append(acc)
            if len(history['accelerations']) > self.history_size:
                history['accelerations'].pop(0)

    def _predict_position(self, track_id: int) -> np.ndarray:
        """
        预测目标下一个位置
        :return: 预测的边界框 [x1,y1,x2,y2]
        """
        history = self.track_history.get(track_id, None)
        if not history or len(history['positions']) == 0:
            return None
        
        current = history['positions'][-1]
        velocity = history['velocities'][-1] if history['velocities'] else np.zeros(4)
        acceleration = history['accelerations'][-1] if history['accelerations'] else np.zeros(4)
        
        return current + velocity + 0.5 * acceleration

    def _calculate_target_roi(self, track_id: int) -> list:
        """计算单个目标的ROI区域"""
        predicted = self._predict_position(track_id)
        if predicted is None:
            return None
        
        # 计算扩展区域
        x1, y1, x2, y2 = predicted
        w = x2 - x1
        h = y2 - y1
        expanded = [
            max(0, x1 - w * self.roi_expansion),
            max(0, y1 - h * self.roi_expansion),
            min(self.frame_width, x2 + w * self.roi_expansion),
            min(self.frame_height, y2 + h * self.roi_expansion)
        ]
        
        # 添加边界缓冲
        return [
            max(0, expanded[0] - self.frame_buffer),
            max(0, expanded[1] - self.frame_buffer),
            min(self.frame_width, expanded[2] + self.frame_buffer),
            min(self.frame_height, expanded[3] + self.frame_buffer)
        ]

    def _merge_rois(self, rois: list) -> np.ndarray:
        """合并所有ROI区域"""
        if not rois:
            return np.array([[0, 0, self.frame_width, self.frame_height]])
        
        return np.array([[
            min(r[0] for r in rois),
            min(r[1] for r in rois),
            max(r[2] for r in rois),
            max(r[3] for r in rois)
        ]])
    
    def process_frame(self, frame: np.ndarray) -> dict:
        """
        处理视频帧的核心方法
        :return: 包含跟踪结果和ROI的字典
        """
        self.frame_number += 1
        
        # 更新帧尺寸
        if self.current_roi is None:
            start_time = time.time()
            self._initialize_roi()
            end_time = time.time()
            self.timing_stats["initialize_roi"].append((end_time - start_time)*1000)
        
        # ROI裁剪
        start_time = time.time()
        x1, y1, x2, y2 = self.current_roi[0].astype(int)
        roi_frame = frame[y1:y2, x1:x2]
        end_time = time.time()
        self.timing_stats["roi_crop"].append((end_time - start_time)*1000)
        
        # YOLO检测
        start_time = time.time()
        results = self.detector(roi_frame, classes=0, conf=0.3, verbose=False, half=True, imgsz=640)  # 只检测人物
        end_time = time.time()
        self.timing_stats["model_inference"].append((end_time - start_time)*1000)
        
        # 转换检测坐标到原图
        start_time = time.time()
        detections = []
        for result in results:
            for box in result.boxes.xyxy.cpu().numpy():
                global_box = box.copy()
                global_box[[0, 2]] += x1
                global_box[[1, 3]] += y1
                confidence = float(result.boxes.conf[0].item())
                detections.append(np.array([*global_box, confidence]))  # 添加置信度
        end_time = time.time()
        self.timing_stats["convert_coordinates"].append((end_time - start_time)*1000)
        
        # SORT跟踪
        start_time = time.time()
        if len(detections) > 0:
            tracked = self.tracker.update(np.array(detections), np.array([]))
        else:
            tracked = self.tracker.update(np.empty((0, 5)), np.array([]))
        end_time = time.time()
        self.timing_stats["sort_tracking"].append((end_time - start_time)*1000)
        
        # 更新历史数据
        start_time = time.time()
        for t in tracked:
            track_id = int(t[4])
            bbox = t[:4].astype(int)
            self._update_history(track_id, bbox)
        end_time = time.time()
        self.timing_stats["update_history"].append((end_time - start_time)*1000)
        
        # 计算各目标ROI
        start_time = time.time()
        active_rois = [self._calculate_target_roi(tid) for tid in self.track_history]
        valid_rois = [r for r in active_rois if r is not None]
        end_time = time.time()
        self.timing_stats["calculate_target_roi"].append((end_time - start_time)*1000)
        
        # 合并ROI并更新
        start_time = time.time()
        self.current_roi = self._merge_rois(valid_rois)
        end_time = time.time()
        self.timing_stats["merge_rois"].append((end_time - start_time)*1000)
        
        tracked_players = []
        for t in tracked:
            track_id = int(t[4])
            bbox = t[:4].astype(int).tolist()
            tracked_players.append({
                'id': track_id,
                'bbox': bbox
            })
        # print(tracked_players)
        # 返回结构化数据
        return {
            'frame_number': self.frame_number,
            'current_roi': self.current_roi[0].astype(int).tolist(),
            'tracked_players': tracked_players
        }

    def visualize(self, frame: np.ndarray) -> np.ndarray:
        """可视化跟踪结果"""
        # 绘制ROI
        x1, y1, x2, y2 = self.current_roi[0].astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # 绘制跟踪框
        for t in self.track_history.values():
            print(t['positions'])
            if len(t['positions']) > 0:
                x1, y1, x2, y2 = t['positions'][-1].astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        return frame
    
    def print_timing_summary(self):
        print("\n--- Timing Summary (Average per call) ---")
        for key, times in self.timing_stats.items():
            if times:
                avg_time = sum(times) / len(times)
                print(f"{key}: {avg_time:.6f} ms")
            else:
                print(f"{key}: No calls recorded")
        print("----------------------------------------")

# # 使用示例
if __name__ == "__main__":
    tracker = PlayerTracker()
    cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # 处理帧并获取结果
        result = tracker.process_frame(frame)
        
        # # 可视化
        
        vis_frame = tracker.visualize(frame)
        cv2.imshow('Tracking', vis_frame)
        
        if cv2.waitKey(1) == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    tracker.print_timing_summary()
    

[array([574, 223, 593, 270])]
[array([ 995,  478, 1057,  663])]
[array([574, 223, 593, 270]), array([574, 224, 592, 269])]
[array([ 995,  478, 1057,  663]), array([ 996,  479, 1055,  658])]
[array([574, 223, 593, 270]), array([574, 224, 592, 269]), array([574, 223, 592, 268])]
[array([ 995,  478, 1057,  663]), array([ 996,  479, 1055,  658]), array([ 995,  473, 1054,  655])]
[array([574, 223, 593, 270]), array([574, 224, 592, 269]), array([574, 223, 592, 268]), array([573, 223, 592, 269])]
[array([ 995,  478, 1057,  663]), array([ 996,  479, 1055,  658]), array([ 995,  473, 1054,  655]), array([ 993,  471, 1053,  654])]
[array([574, 223, 593, 270]), array([574, 224, 592, 269]), array([574, 223, 592, 268]), array([573, 223, 592, 269]), array([573, 223, 592, 269])]
[array([ 995,  478, 1057,  663]), array([ 996,  479, 1055,  658]), array([ 995,  473, 1054,  655]), array([ 993,  471, 1053,  654]), array([ 996,  472, 1053,  650])]
[array([574, 223, 593, 270]), array([574, 224, 592, 269]), a

In [12]:
import cv2
import numpy as np
import os
import sys

sys.path.append('/home/max/Desktop/tennis/tennis-v5')

from ultralytics import YOLO
from algorithms.sort import Sort
from utils.shared_data import SharedData
import torchvision.transforms as transforms

import time

model = YOLO('/home/max/Desktop/tennis/tennis-v5/weights/yolo11x.pt').to('cuda')


cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

timing_stats = []

while True:
    ret, frame = cap.read()
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (640,640), swapRB=True)
    if not ret:
        break
    start_time = time.time()
    results = model(blob, classes=0, conf=0.3, verbose=False, half=True, imgsz=640)
    end_time = time.time()
    timing_stats.append((end_time - start_time)*1000)
    # for result in results:
    #     for box in result.boxes.xyxy.cpu().numpy():
    #         cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 0, 255), 2)
    
#     cv2.imshow('Tracking', frame)
#     if cv2.waitKey(1) == ord('q'):
#         break

cap.release()
# cv2.destroyAllWindows()
print(f"Average time taken: {sum(timing_stats)/len(timing_stats):.6f} ms")


error: OpenCV(4.10.0) /home/conda/feedstock_root/build_artifacts/libopencv_1735816972891/work/modules/imgproc/src/resize.cpp:3789: error: (-215:Assertion failed) !dsize.empty() in function 'resize'


In [ ]:
for track_id in lost_tracks:
    last_pos = track.history[-1]
    next_pos = track.predicted_pos
    for i in range(lost_frames):
        interp_pos = last_pos + (next_pos - last_pos) * (i+1)/lost_frames
        trajectories[track_id].append(interp_pos)

In [2]:
import cv2
import numpy as np
import pandas as pd
# import time
from ultralytics import YOLO

class PersonDetector:
    def __init__(self):
        self.model = YOLO('yolo11x.pt')
        self.min_score = 0.4
        self.player1_boxes = []
        self.player2_boxes = []
        self.height = None
        self.width = None
        self.roi_mask = None  # 添加ROI掩码属性

    @staticmethod
    def calculate_iou(boxA, boxB):
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
        areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
        return inter / float(areaA + areaB - inter) if (areaA + areaB - inter) > 0 else 0

    def set_roi_mask(self, mask):
        """设置感兴趣区域(ROI)掩码"""
        self.roi_mask = mask
        print(f"ROI掩码已设置: 掩码大小 {mask.shape}")
        return self

    def apply_roi(self, frame):
        # 如果存在外部设置的ROI掩码，优先使用它
        if self.roi_mask is not None and self.roi_mask.shape[:2] == frame.shape[:2]:
            # 确保掩码维度匹配
            mask = self.roi_mask.copy()
        else:
            # 否则使用默认的ROI生成逻辑
            mask = np.zeros(frame.shape[:2], dtype=np.uint8)
            if not self.player1_boxes and not self.player2_boxes:
                cv2.rectangle(mask, (int(self.width*0.15), int(self.height*0.1)), 
                             (int(self.width*0.85), int(self.height*0.4)), 255, -1)
                cv2.rectangle(mask, (int(self.width*0.15), int(self.height*0.5)), 
                             (int(self.width*0.85), int(self.height*0.9)), 255, -1)
            else:
                if self.player1_boxes:
                    last_p1 = self.player1_boxes[-1]
                    if np.any(last_p1):
                        x1,y1,x2,y2 = last_p1.astype(int)
                        margin = 650
                        cv2.rectangle(mask, 
                                    (max(0,x1-margin), max(0,y1-margin)),
                                    (min(self.width,x2+margin), min(self.height,y2+margin)),
                                    255, -1)
        
        # 应用掩码到原始帧
        return cv2.bitwise_and(frame, frame, mask=mask)

    def detect(self, frame, frame_number):
        self.height, self.width = frame.shape[:2]
        processed = self.apply_roi(frame)
        results = self.model(processed, verbose=False)[0]
        
        boxes = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()
        persons = boxes[(results.boxes.cls.cpu().numpy() == 0) & (scores > self.min_score)]
        
        if not self.player1_boxes and not self.player2_boxes:
            lower_boxes = [b for b in persons if b[3] > self.height/2]
            upper_boxes = [b for b in persons if (b[3] <= self.height*0.4) & (b[1] >= self.height*0.1)]
            p1 = max(lower_boxes, key=lambda x: x[3], default=np.zeros(4))
            p2 = max(upper_boxes, key=lambda x: x[3], default=np.zeros(4))
        else:
            p1 = self.player1_boxes[-1].copy() if self.player1_boxes else np.zeros(4)
            p2 = self.player2_boxes[-1].copy() if self.player2_boxes else np.zeros(4)
            max_iou1, max_iou2 = 0.3, 0.3
            
            for box in persons:
                if self.player1_boxes:
                    iou1 = self.calculate_iou(self.player1_boxes[-1], box)
                    if iou1 > max_iou1:
                        max_iou1 = iou1
                        p1 = box
                if self.player2_boxes:
                    iou2 = self.calculate_iou(self.player2_boxes[-1], box)
                    if iou2 > max_iou2:
                        max_iou2 = iou2
                        p2 = box
        
        self.player1_boxes.append(p1)
        self.player2_boxes.append(p2)
        
        return [frame_number, *map(int, p1), *map(int, p2)]

detector = PersonDetector()
cap = cv2.VideoCapture('/home/max/Desktop/tennis/tennis-v5/data/InputVideos/input.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
out = cv2.VideoWriter('/home/max/Desktop/tennis/tennis-v5/data/OutputVideos/output_person.mp4', fourcc, fps, (width, height))
results = []
frame_number = 0

# start_time = time.time()
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    result = detector.detect(frame, frame_number)
    results.append(result)
    
    cv2.rectangle(frame, (result[1], result[2]), (result[3], result[4]), (0,255,0), 2)
    cv2.rectangle(frame, (result[5], result[6]), (result[7], result[8]), (0,0,255), 2)
    out.write(frame)
    frame_number += 1

cap.release()
out.release()
pd.DataFrame(results, columns=['frame_number','x1','y1','x2','y2','x3','y3','x4','y4']).to_csv('/home/max/Desktop/tennis/tennis-v3/models/1_combined_output.csv', index=False)
# print(f"Processed in {time.time()-start_time:.2f}s, {frame_number/(time.time()-start_time):.2f}fps")


100%|██████████| 109M/109M [00:12<00:00, 9.38MB/s] 
